# 05 · Prompt 1: the most similar properties

**Purpose.** State how we answer the brief's first example prompt, and (once `similarity.py` is complete) show and
check the answers for each marketed listing.

The evidence behind each choice is in the similarity notebooks: [`02_similarity_build_year`](02_similarity_build_year.ipynb),
[`03_similarity_location`](03_similarity_location.ipynb) and [`04_similarity_physical`](04_similarity_physical.ipynb).

## 1. The question

> "Find the 10 most similar properties in the estate to the newly marketed property."

How we read it:

* **The input is one marketed listing.** Each listing (M1-M5) is a single unit, e.g. M3 is Spitfire Park, Unit 42.
* **The answer is 10 units from Indurent's portfolio** ("the estate"): the 1,250 units most like that listing.
* **"Properties" means units, and the answer is the literal top 10: no cap on units per estate.** Units on the
  estate a listing sits on genuinely are among the most similar (same location, often similar age and size), and
  every listing with coordinates sits on an existing estate (notebook 03). So a top 10 can be concentrated: in a
  first rough ranking, all 10 of M3's are on Velvet Park 60, its own estate. The answer says so when it happens
  (*"all 10 are on Velvet Park 60, the estate M3 sits on"*).
* **A cap is available on request.** If the user asks for a spread ("similar units on other estates", "at most 2
  per estate"), the ranking can limit units per estate. It's off by default because it changes the answer away
  from "the 10 most similar". (EDA section 4.3 first suggested a cap; this replaces that suggestion.)
* **"The newly marketed property" is singular, but there are five listings.** The answer either asks which one is
  meant or answers for all five; it never picks one silently.

## 2. How similarity is defined

Each portfolio unit gets **three separate similarity scores** against the listing, one per block, each between 0
and 1. They use the same form, `0.5 ** (gap / half-life)`: 1 for an identical value, halving with every half-life of
difference, and the same whichever direction the difference goes.

| Block | Compares | 1 when | Halves every | Evidence |
|---|---|---|---|---|
| **Age** | build years | built the same year | 10 years apart (roughly one era of building specification) | notebook 02 |
| **Location** | coordinates (straight-line miles) | in the same spot | 10 miles apart (a local industrial market) | notebook 03 |
| **Physical** | size (sq ft, log scale) | the same size | each doubling (or halving) in size | notebook 04 |

**Why size uses a ratio, while age and location use a fixed gap.** Each half-life is measured in the kind of
difference that matters for that quantity:

* **Location and age: an absolute gap.** 10 miles apart means much the same anywhere, and 10 years between 1975 and
  1985 is about as big a step in building specification as 10 years between 2000 and 2010.
* **Size: a relative gap.** The same 1,000 sq ft difference means very different things: 1,000 vs 2,000 sq ft is
  twice the size (a different class of unit), while 100,000 vs 101,000 sq ft is 1% bigger (effectively the same).
  A fixed sq ft half-life would get one of these badly wrong; a ratio treats 1k vs 2k the same as 50k vs 100k, each
  a step up to a different class of building. It's the same reason size is plotted on a log scale.

**Physical means size only.** Size is the only physical field every listing has; eaves exist for M4 and M5 only, and
usable door counts for 39 portfolio units. Eaves, doors and volume are shown as context, not scored (notebook 04).

**Combining the blocks into one ranking:**

* **Overall score = the average of the three blocks, equally weighted** by default; the weights can be changed.
* **A block the listing doesn't have is dropped** for that listing, and the answer says so: M2 has no build year or
  coordinates, so it's matched on size only; M5 is offshore, so it's matched on age and size.
* **A block a portfolio unit doesn't have scores 0**, so a unit we know less about can't outrank one that matches on
  everything. 358 units have no usable location and 16 no build year; they can still appear, marked
  "unknown" on that block.
* **No cap on units per estate** by default; an optional cap if the user asks for a spread.
* **All three block scores are kept**, not just the average, so an answer can say *why* a unit is similar, e.g. "on
  the same estate and a similar size, but 25 years newer".

Which blocks each listing can be matched on:

In [1]:
from pathlib import Path

import pandas as pd

from indurent_matching.data import load_marketed, load_portfolio

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

blocks = pd.DataFrame({
    "age": marketed["build_year"].notna() & ~marketed["build_year_doubtful"],
    "location": marketed["location_confidence"] != "unknown",
    "physical": marketed["size_sqft"].notna() & ~marketed["size_sqft_doubtful"],
}, index=marketed.index)
blocks.insert(0, "listing", marketed["industrial_estate_name"] + ", " + marketed["unit_name"])
blocks["matched on"] = blocks[["age", "location", "physical"]].apply(
    lambda row: ", ".join(block for block, ok in row.items() if ok), axis=1)
blocks.replace({True: "yes", False: "-"})

,listing,age,location,physical,matched on
listing_id,,,,,
M1,"Cherry Lane, Unit 10",yes,yes,yes,"age, location, physical"
M2,"Tech Hub, Unit 8",-,-,yes,physical
M3,"Spitfire Park, Unit 42",yes,yes,yes,"age, location, physical"
M4,"Stable Lane, Unit 14",yes,yes,yes,"age, location, physical"
M5,"Chancery Depot, Unit 2",yes,-,yes,"age, physical"


In [2]:
# The same check on the portfolio side: units missing a block score 0 on it.
pd.Series({
    "no usable build year": (portfolio["build_year"].isna() | portfolio["build_year_doubtful"]).sum(),
    "no usable location": (portfolio["location_confidence"] == "unknown").sum(),
    "no usable size": (portfolio["size_sqft"].isna() | portfolio["size_sqft_doubtful"]).sum(),
}, name="portfolio units").to_frame()

,portfolio units
no usable build year,16
no usable location,358
no usable size,10


* **M1, M3 and M4** can be matched on all three blocks.
* **M2 (Tech Hub)** only on physical (size): it has no build year and no coordinates.
* **M5 (Chancery Depot)** on age and physical: its coordinates are 27 km offshore.
* On the portfolio side, most units have a usable build year and size; **358 have no usable location**, so they
  score 0 on location when a listing is matched on it.

## 3. The top 10 for each listing

`most_similar` from `similarity.py`, with the defaults: equal weights, no cap on units per estate. Block scores are
shown blank where the unit has no usable value (it counted 0 in `overall`) or where the listing can't be compared
on that block.

In [3]:
from contextlib import contextmanager

import indurent_matching.similarity as similarity
from indurent_matching.similarity import most_similar


SHOW = ["industrial_estate_name", "unit_name", "overall", "age", "location", "physical",
        "build_year", "size_sqft", "miles_from_listing"]
top10 = {listing_id: most_similar(row, portfolio) for listing_id, row in marketed.iterrows()}


def show(listing_id):
    row = marketed.loc[listing_id]
    print(f"{listing_id} {row['industrial_estate_name']}, {row['unit_name']}: built {row['build_year']:.0f}, "
          f"{row['size_sqft']:,.0f} sq ft, matched on {blocks.loc[listing_id, 'matched on']}"
          if pd.notna(row["build_year"]) else
          f"{listing_id} {row['industrial_estate_name']}, {row['unit_name']}: {row['size_sqft']:,.0f} sq ft, "
          f"matched on {blocks.loc[listing_id, 'matched on']}")
    return top10[listing_id][SHOW].round({"overall": 2, "age": 2, "location": 2, "physical": 2,
                                          "size_sqft": 0, "miles_from_listing": 1})


show("M1")

M1 Cherry Lane, Unit 10: built 1977, 16,860 sq ft, matched on age, location, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
586,Quartz Wharf 16,Unit 5,0.88,0.93,0.73,0.97,1976.0,17340.0,4.5
584,Quartz Wharf 16,Unit 3,0.81,0.87,0.74,0.82,1975.0,20565.0,4.4
1157,Scarlet Pavilion 78,Unit 7,0.76,0.76,1.00,0.52,1981.0,8752.0,0.1
606,Quartz Wharf 16,Unit 25,0.72,0.47,0.74,0.96,1988.0,16208.0,4.4
612,Quartz Wharf 16,Unit 31,0.69,0.76,0.73,0.57,1981.0,9603.0,4.5
642,Quartz Wharf 16,Unit 61,0.67,0.76,0.73,0.53,1973.0,8945.0,4.5
673,Ruby Junction 67,Unit 7,0.65,1.00,NaN,0.95,1977.0,17793.0,NaN
1153,Scarlet Pavilion 78,Unit 3,0.64,0.81,1.00,0.12,1974.0,2011.0,0.0
418,Verdant Oaks 7,Unit 9,0.64,1.00,NaN,0.91,1977.0,15273.0,NaN


In [4]:
show("M2")

M2 Tech Hub, Unit 8: 906 sq ft, matched on physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
602,Quartz Wharf 16,Unit 21,1.00,NaN,NaN,1.00,1979.0,905.0,NaN
689,Ruby Junction 67,Unit 23,1.00,NaN,NaN,1.00,2006.0,905.0,NaN
221,Opal Hub 4,Unit 2,1.00,NaN,NaN,1.00,2022.0,905.0,NaN
874,Velvet Park 60,Unit 29,1.00,NaN,NaN,1.00,2004.0,902.0,NaN
698,Ruby Junction 67,Unit 32,1.00,NaN,NaN,1.00,1998.0,901.0,NaN
24,Sable Mill 91,Unit 6,0.99,NaN,NaN,0.99,2016.0,898.0,NaN
1101,Stone Mews 81,Unit 4,0.98,NaN,NaN,0.98,1986.0,889.0,NaN
601,Quartz Wharf 16,Unit 20,0.98,NaN,NaN,0.98,2017.0,923.0,NaN
863,Velvet Park 60,Unit 18,0.98,NaN,NaN,0.98,1976.0,923.0,NaN


In [5]:
show("M3")

M3 Spitfire Park, Unit 42: built 1976, 2,294 sq ft, matched on age, location, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
869,Velvet Park 60,Unit 24,0.84,0.81,0.99,0.71,1973.0,1638.0,0.1
865,Velvet Park 60,Unit 20,0.81,0.76,1.00,0.69,1980.0,3332.0,0.0
863,Velvet Park 60,Unit 18,0.80,1.00,1.00,0.40,1976.0,923.0,0.1
855,Velvet Park 60,Unit 10,0.80,0.87,1.00,0.52,1978.0,1199.0,0.1
878,Velvet Park 60,Unit 33,0.77,0.66,0.99,0.66,1970.0,1520.0,0.1
849,Velvet Park 60,Unit 4,0.77,0.44,1.00,0.88,1988.0,2013.0,0.0
851,Velvet Park 60,Unit 6,0.74,0.93,0.99,0.29,1977.0,675.0,0.1
860,Velvet Park 60,Unit 15,0.73,0.76,0.99,0.45,1980.0,5086.0,0.1
852,Velvet Park 60,Unit 7,0.73,0.41,1.00,0.79,1989.0,2904.0,0.0


In [6]:
show("M4")

M4 Stable Lane, Unit 14: built 2000, 617 sq ft, matched on age, location, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
26,Sable Mill 91,Unit 8,0.70,0.54,1.00,0.56,2009.0,1098.0,0.0
27,Sable Mill 91,Unit 9,0.69,0.76,0.99,0.31,2004.0,1985.0,0.1
25,Sable Mill 91,Unit 7,0.68,0.22,0.99,0.84,2022.0,734.0,0.1
24,Sable Mill 91,Unit 6,0.67,0.33,1.00,0.69,2016.0,898.0,0.0
20,Sable Mill 91,Unit 2,0.66,0.93,1.00,0.05,1999.0,11839.0,0.0
871,Velvet Park 60,Unit 26,0.66,1.00,0.00,0.97,2000.0,639.0,118.1
88,Scarlet Grove 78,Unit 7,0.65,0.93,0.09,0.93,2001.0,667.0,34.8
21,Sable Mill 91,Unit 3,0.64,0.76,1.00,0.18,2004.0,3479.0,0.1
844,Ruby Arc 96,Unit 47,0.62,0.87,0.00,0.99,2002.0,613.0,116.1


In [7]:
show("M5")

M5 Chancery Depot, Unit 2: built 1966, 16,614 sq ft, matched on age, physical


,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
482,Sapphire Grove 50,Unit 10,0.93,1.00,NaN,0.87,1966.0,14412.0,NaN
478,Sapphire Grove 50,Unit 6,0.87,0.76,NaN,0.97,1962.0,17052.0,NaN
810,Ruby Arc 96,Unit 13,0.84,0.71,NaN,0.98,1971.0,17017.0,NaN
258,Steel Junction 81,Unit 4,0.81,0.71,NaN,0.91,1971.0,15141.0,NaN
656,Onyx Beacon 83,Unit 4,0.80,0.93,NaN,0.67,1965.0,11101.0,NaN
918,Crimson Mill 42,Unit 1,0.78,0.71,NaN,0.86,1971.0,19332.0,NaN
211,Scarlet Depot 92,Unit 7,0.78,0.76,NaN,0.81,1970.0,13429.0,NaN
53,Ruby Mill 13,Unit 4,0.78,0.76,NaN,0.80,1970.0,20683.0,NaN
210,Scarlet Depot 92,Unit 6,0.78,1.00,NaN,0.56,1966.0,9221.0,NaN


**What the top 10s look like:**

* **M1 (Cherry Lane, Wales, 1977, 16,860 sq ft)**: led by **Quartz Wharf 16**, the estate 4.4 miles away, where
  several units match closely on age and size (Unit 5: 1976, 17,340 sq ft, score 0.88). Two units on Scarlet
  Pavilion 78, the estate M1 sits on, also appear. Three of the ten have no usable location (blank), but match
  on age and size almost exactly.
* **M2 (Tech Hub, 906 sq ft)**: matched on size alone, so the top 10 are simply the units closest to 906 sq ft,
  built anywhere between 1976 and 2022.
* **M3 (Spitfire Park, Midlands, 1976, 2,294 sq ft)**: all ten are on **Velvet Park 60**, the estate it sits on. They
  score about 1 on location, and the order within them is set by age and size.
* **M4 (Stable Lane, 2000, 617 sq ft)**: seven on **Sable Mill 91**, its own estate, plus three tiny units built
  around 2000 elsewhere, two of them over 100 miles away. With equal weights, a unit that matches perfectly on age
  and size can rank in the top 10 from anywhere (location 0, but 1 + 1 out of 3 = 0.67).
* **M5 (Chancery Depot, offshore, 1966, 16,614 sq ft)**: matched on age and size only, so the top 10 are mid-sized
  units built 1962-1971, from eight estates around the country.

## 4. Checks on the ranking

**Where the top 10 come from, and which block they score highest on.** For each listing: how many of its top 10
are on the estate it sits on, how many estates they span, and the average of each block across the top 10.

In [8]:
from indurent_matching.similarity import sits_on_estate

summary = pd.DataFrame({
    listing_id: {
        "sits on": sits_on_estate(marketed.loc[listing_id], portfolio),
        "top 10 on that estate": (t["industrial_estate_name"] == sits_on_estate(marketed.loc[listing_id], portfolio)).sum(),
        "estates in top 10": t["industrial_estate_name"].nunique(),
        "mean overall": t["overall"].mean(),
        "mean age": t["age"].mean(),
        "mean location": t["location"].mean(),
        "mean physical": t["physical"].mean(),
        "location unknown in top 10": t["location"].isna().sum() if blocks.loc[listing_id, "location"] else None,
    }
    for listing_id, t in top10.items()
}).T
summary.round(2)

,sits on,top 10 on that estate,estates in top 10,mean overall,mean age,mean location,mean physical,location unknown in top 10
M1,Scarlet Pavilion 78,2,5,0.708976,0.835593,0.809553,0.724649,3
M2,NaN,0.0,7.0,0.990428,NaN,NaN,0.990428,NaN
M3,Velvet Park 60,10,1,0.772911,0.716859,0.996157,0.605716,0
M4,Sable Mill 91,7,4,0.65912,0.687162,0.706468,0.583731,0
M5,NaN,0.0,8.0,0.815316,0.803503,NaN,0.827129,NaN


* **Location dominates when a listing sits on an estate**: M3's ten are all on its own estate, M4's mostly. That's
  the expected result of no cap (section 1): units on the same estate really are the closest comparables.
* **M1 is different**: its own estate (Scarlet Pavilion 78) is small and its units differ in size from M1, so a
  nearby bigger estate (Quartz Wharf 16) supplies most of the matches.
* **Unknown locations do appear** (three of M1's ten). They score 0 on location, so they need near-perfect age and
  size to get in, and an answer should flag them as "location unknown".

**Sensitivity to the half-lives.** Each half-life is halved and doubled in turn, and the table shows how many
of each listing's default top 10 survive. 10 means the choice doesn't matter for that listing; a low number means
the ranking is sensitive to it.

In [9]:
HALF_LIVES = {"age": "AGE_HALF_LIFE_YEARS", "location": "LOCATION_HALF_LIFE_MILES", "physical": "SIZE_HALF_LIFE_RATIO"}


@contextmanager
def half_life(block, factor):
    """Temporarily scale one block's half-life (for size, the ratio is raised to the factor: 2 -> 4 or 1.41)."""
    name = HALF_LIVES[block]
    original = getattr(similarity, name)
    setattr(similarity, name, original**factor if block == "physical" else original * factor)
    try:
        yield
    finally:
        setattr(similarity, name, original)


rows = {}
for block in HALF_LIVES:
    for factor, label in [(0.5, "halved"), (2, "doubled")]:
        with half_life(block, factor):
            rows[f"{block} {label}"] = {
                listing_id: len(set(most_similar(row, portfolio).index) & set(top10[listing_id].index))
                for listing_id, row in marketed.iterrows()
            }
pd.DataFrame(rows).T

,M1,M2,M3,M4,M5
age halved,9,10,8,9,8
age doubled,8,10,8,9,8
location halved,9,10,10,10,10
location doubled,8,10,8,8,10
physical halved,9,10,8,9,8
physical doubled,7,10,9,9,8


* **The rankings are fairly stable**: halving or doubling any one half-life keeps 7-10 of a listing's 10 matches.
  So the exact choices (10 years, 10 miles, twice the size) shape the results but don't decide them.
* **Location's half-life matters most when it's doubled.** Halving it (5 miles) changes almost nothing: M3's ten
  stay on Velvet Park 60. Doubling it (20 miles) swaps two units each for M1, M3 and M4, as estates 15-25 miles
  away start to count: M3 gains one unit each on Granite Pointe 39 and Cedar Pavilion 55. M5 isn't matched on
  location at all.
* **M2 never changes**: with size alone, the order of units doesn't depend on how fast similarity falls off.

**The optional cap.** M3's top 10 with at most 2 units per estate, for when the user asks for a spread:

In [10]:
most_similar(marketed.loc["M3"], portfolio, max_per_estate=2)[SHOW].round(
    {"overall": 2, "age": 2, "location": 2, "physical": 2, "size_sqft": 0, "miles_from_listing": 1})

,industrial_estate_name,unit_name,overall,age,location,physical,build_year,size_sqft,miles_from_listing
property_id,,,,,,,,,
869,Velvet Park 60,Unit 24,0.84,0.81,0.99,0.71,1973.0,1638.0,0.1
865,Velvet Park 60,Unit 20,0.81,0.76,1.00,0.69,1980.0,3332.0,0.0
713,Sapphire Oaks 10,Unit 15,0.67,1.00,NaN,1.00,1976.0,2299.0,NaN
465,Granite Pointe 39,Unit 31,0.66,0.76,0.23,0.99,1980.0,2323.0,21.4
299,Cedar Pavilion 55,Unit 5,0.65,0.87,0.29,0.80,1978.0,1828.0,17.8
1059,Ruby Bridge 60,Unit 4,0.64,0.93,0.00,1.00,1977.0,2295.0,179.5
87,Scarlet Grove 78,Unit 6,0.64,0.93,0.00,0.97,1977.0,2355.0,152.8
710,Sapphire Oaks 10,Unit 12,0.63,0.93,NaN,0.97,1977.0,2219.0,NaN
769,Crimson Forge 9,Unit 9,0.63,0.93,0.00,0.94,1975.0,2167.0,96.0


* With the cap, M3's list keeps its two best Velvet Park 60 units, then spreads out: units built 1975-1980 of
  almost exactly M3's size, on estates 18-180 miles away or with no usable location.
* That's the "show me similar units on other estates" answer: useful on request, but not the literal top 10.

**M2 is matched on size alone, so many units tie.** How many portfolio units score as well as M2's 10th
match (to two decimal places)?

In [11]:
m2_all = most_similar(marketed.loc["M2"], portfolio, k=len(portfolio))
tenth = m2_all["overall"].iloc[9]
print(f"M2's 10th match scores {tenth:.3f}; {(m2_all['overall'].round(2) >= round(tenth, 2)).sum()} units score "
      f"{round(tenth, 2):.2f} or more")

M2's 10th match scores 0.979; 11 units score 0.98 or more


* **11 units score 0.98 or more for M2**, so which ten make its list comes down to tiny size differences (and
  Property Id for exact ties). The answer should say M2 is matched on size only, and that many units are about as
  close.

## 5. Summary

* `most_similar` returns sensible comparables: on the estate a listing sits on when its units match, from nearby
  estates when they don't (M1), and on age and size alone when there's no usable location (M5) or only size (M2).
* The ranking is reasonably stable to the half-life choices.
* Things answers should state: which blocks a listing was matched on; when the top 10 is concentrated on one
  estate; units with an unknown location; and, for size-only matches, that many units tie.
* **Weights stay equal.** Equal weights let a unit 100+ miles away into the top 10 if its age and size match
  perfectly (M4's #6 and #9), but there's no evidence in the data for favouring one block, and the brief asks for
  all three dimensions. A user who cares more about location can say so, and the weights change for that answer
  (`weights={"location": 2}`); the answer should point such distant units out.